# Visualize a Single Raw DNS Snapshot (with Interpolation)

This notebook loads a **single**, unprocessed 3D snapshot from a raw binary file, interpolates 2D slices onto a uniform grid, and visualizes them. It is designed for lightweight, interactive exploration of the spatial structure of your data at one moment in time.

**Workflow:**
1.  Set all parameters in the **User Configuration** cell.
2.  Run the **Load Snapshot Data** cell to load the binary file into memory.
3.  Modify slice indices in the parameter cells and run the individual plot cells as needed.

### 1. Setup and Imports

In [ ]:
import numpy as np
from pathlib import Path
import logging

# Import only the spatial slice plotting functions
from mhd_surrogate_core.plotting import (
    plot_interpolated_xz_slice,
    plot_interpolated_xy_slice,
    plot_interpolated_yz_slice
)

### 2. User Configuration

**Action Required:** Set your parameters in this cell. You can re-run this cell to apply changes without reloading the data.

In [ ]:
# <<< 1. SET SNAPSHOT FILE PATH AND PARAMETERS >>>
snapshot_file_path = Path("/raid/skowronek/preprocessed_dns_output/01-Cold_Runs/01-Re16K_Ha325/raw/patt3d_vx3d_000609")

nx, ny, nz = 2301, 481, 121  # Grid dimensions for the snapshot
source_channel_labels = ['vx', 'vy', 'vz', 'T'] # IMPORTANT: Must be in file order

# <<< 2. DEFINE CHANNEL NAME MAPPING >>>
channel_map = {
    'vx': 'u',
    'vy': 'v',
    'vz': 'w'
}

# <<< 3. SET INTERPOLATION GRID SIZE >>>
num_interp_points_y = 1024
num_interp_points_z = 1024

# <<< 4. SET GLOBAL PLOT SIZING AND UNIT PARAMETERS >>>
global_base_size = 20
global_min_size = 2
global_unit_label = ""

# <<< 5. SET GLOBAL COLOR SCALE OVERRIDE (OPTIONAL) >>>
global_vmin_override = -5
global_vmax_override = 6

print("Configuration set. You can now run the 'Load Snapshot Data' cell.")

### 3. Load Snapshot Data

**Run this cell only once** to load the raw binary data from the specified file.

In [ ]:
# --- Data Loading and Initialization ---
snapshot_data_3d = None
raw_coords = {}

if not snapshot_file_path.exists():
    logging.error(f"ERROR: Snapshot file not found at {snapshot_file_path}. Please set the correct path.")
else:
    try:
        input_dtype = np.float64
        with open(snapshot_file_path, 'rb') as f:
            # 1. Read coordinates
            raw_coords['x'] = np.fromfile(f, dtype=input_dtype, count=nx)
            raw_coords['y'] = np.fromfile(f, dtype=input_dtype, count=ny)
            raw_coords['z'] = np.fromfile(f, dtype=input_dtype, count=nz)
            raw_coords['labels'] = source_channel_labels
            # 2. Read all channel data
            channel_data_1d = np.fromfile(f, dtype=input_dtype)
        
        # 3. Reshape and transpose the data to a logical (x, y, z, channel) layout
        num_input_channels = len(source_channel_labels)
        data_4d_physical = channel_data_1d.reshape((nz, num_input_channels, ny, nx))
        snapshot_data_3d = data_4d_physical.transpose(3, 2, 0, 1).astype(np.float32)
        
        print("Snapshot data loaded successfully into memory.")
        print(f"Final data shape: {snapshot_data_3d.shape} (x, y, z, channel)")
    except Exception as e:
        logging.error(f"Failed to load or process snapshot file: {e}")
        logging.error("Please check grid dimensions (nx, ny, nz) and source channel labels.")

# Identify velocity components to be plotted
velocity_components = sorted([c for c in raw_coords.get('labels', []) if c in channel_map])
if not velocity_components:
    logging.warning("Warning: No velocity components matching the channel_map found.")

### 4. Calculate Global Color Scale

Run this cell to determine the color scale that will be applied to **all** plots in this notebook.

In [ ]:
# --- Calculate the global vmin and vmax across all velocity components ---
global_vmin, global_vmax = None, None

if global_vmin_override is not None and global_vmax_override is not None:
    global_vmin = global_vmin_override
    global_vmax = global_vmax_override
    print(f"Using user-defined global color scale: [{global_vmin:.3f}, {global_vmax:.3f}]")
elif snapshot_data_3d is not None and velocity_components:
    all_vc_data = []
    for vc in velocity_components:
        vc_idx = raw_coords['labels'].index(vc)
        all_vc_data.append(snapshot_data_3d[..., vc_idx])
    
    global_vmin = min(d.min() for d in all_vc_data)
    global_vmax = max(d.max() for d in all_vc_data)
    print(f"Auto-calculated global color scale for all plots: [{global_vmin:.3f}, {global_vmax:.3f}]")

---

### 5. Spatial Slice Plots (2D)

#### 5.1 X-Z Slices (at constant Y)

In [ ]:
# === Parameters for X-Z Slices ===
# <<< MODIFY THESE VALUES >>>
plot_y_index_xz = ny // 2
# ---
print(f"Using y-index {plot_y_index_xz} for X-Z slice plots.")

In [ ]:
# === Plot X-Z Slice for u (vx) ===
if 'vx' in velocity_components:
    plot_interpolated_xz_slice(
        snapshot_data_3d=snapshot_data_3d,
        raw_coords=raw_coords,
        channel='vx',
        y_index=plot_y_index_xz,
        num_interp_points_z=num_interp_points_z,
        channel_alias=channel_map.get('vx'),
        vmin=global_vmin,
        vmax=global_vmax,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

In [ ]:
# === Plot X-Z Slice for v (vy) ===
if 'vy' in velocity_components:
    plot_interpolated_xz_slice(
        snapshot_data_3d=snapshot_data_3d,
        raw_coords=raw_coords,
        channel='vy',
        y_index=plot_y_index_xz,
        num_interp_points_z=num_interp_points_z,
        channel_alias=channel_map.get('vy'),
        vmin=global_vmin,
        vmax=global_vmax,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

In [ ]:
# === Plot X-Z Slice for w (vz) ===
if 'vz' in velocity_components:
    plot_interpolated_xz_slice(
        snapshot_data_3d=snapshot_data_3d,
        raw_coords=raw_coords,
        channel='vz',
        y_index=plot_y_index_xz,
        num_interp_points_z=num_interp_points_z,
        channel_alias=channel_map.get('vz'),
        vmin=global_vmin,
        vmax=global_vmax,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

#### 5.2 X-Y Slices (at constant Z)

In [ ]:
# === Parameters for X-Y Slices ===
# <<< MODIFY THESE VALUES >>>
plot_z_index_xy = nz // 2
# ---
print(f"Using z-index {plot_z_index_xy} for X-Y slice plots.")

In [ ]:
# === Plot X-Y Slice for u (vx) ===
if 'vx' in velocity_components:
    plot_interpolated_xy_slice(
        snapshot_data_3d=snapshot_data_3d,
        raw_coords=raw_coords,
        channel='vx',
        z_index=plot_z_index_xy,
        num_interp_points_y=num_interp_points_y,
        channel_alias=channel_map.get('vx'),
        vmin=global_vmin,
        vmax=global_vmax,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

In [ ]:
# === Plot X-Y Slice for v (vy) ===
if 'vy' in velocity_components:
    plot_interpolated_xy_slice(
        snapshot_data_3d=snapshot_data_3d,
        raw_coords=raw_coords,
        channel='vy',
        z_index=plot_z_index_xy,
        num_interp_points_y=num_interp_points_y,
        channel_alias=channel_map.get('vy'),
        vmin=global_vmin,
        vmax=global_vmax,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

In [ ]:
# === Plot X-Y Slice for w (vz) ===
if 'vz' in velocity_components:
    plot_interpolated_xy_slice(
        snapshot_data_3d=snapshot_data_3d,
        raw_coords=raw_coords,
        channel='vz',
        z_index=plot_z_index_xy,
        num_interp_points_y=num_interp_points_y,
        channel_alias=channel_map.get('vz'),
        vmin=global_vmin,
        vmax=global_vmax,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

#### 5.3 Y-Z Slices (at constant X)

In [ ]:
# === Parameters for Y-Z Slices ===
# <<< MODIFY THESE VALUES >>>
plot_x_index_yz = nx // 2
# ---
print(f"Using x-index {plot_x_index_yz} for Y-Z slice plots.")

In [ ]:
# === Plot Y-Z Slice for u (vx) ===
if 'vx' in velocity_components:
    plot_interpolated_yz_slice(
        snapshot_data_3d=snapshot_data_3d,
        raw_coords=raw_coords,
        channel='vx',
        x_index=plot_x_index_yz,
        num_interp_points_y=num_interp_points_y,
        num_interp_points_z=num_interp_points_z,
        channel_alias=channel_map.get('vx'),
        vmin=global_vmin,
        vmax=global_vmax,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

In [ ]:
# === Plot Y-Z Slice for v (vy) ===
if 'vy' in velocity_components:
    plot_interpolated_yz_slice(
        snapshot_data_3d=snapshot_data_3d,
        raw_coords=raw_coords,
        channel='vy',
        x_index=plot_x_index_yz,
        num_interp_points_y=num_interp_points_y,
        num_interp_points_z=num_interp_points_z,
        channel_alias=channel_map.get('vy'),
        vmin=global_vmin,
        vmax=global_vmax,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

In [ ]:
# === Plot Y-Z Slice for w (vz) ===
if 'vz' in velocity_components:
    plot_interpolated_yz_slice(
        snapshot_data_3d=snapshot_data_3d,
        raw_coords=raw_coords,
        channel='vz',
        x_index=plot_x_index_yz,
        num_interp_points_y=num_interp_points_y,
        num_interp_points_z=num_interp_points_z,
        channel_alias=channel_map.get('vz'),
        vmin=global_vmin,
        vmax=global_vmax,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )